In [0]:

# Live station poller for predictions
# The purpose of this notebook is to build a live but small archive of station status so that we can
# run the predictions in Free Edition rather than trying to rely on writing every 5 minutes from the main site
# to the AWS S3 bucket. So we will repeat that construction here, but actually perform the full pipeline where we
# write to a typed silver table. This isn't meant to create a reproducible archive like the polling in the
# main site is, so we are less careful. 

# Fetches the Citi Bike GBFS station status feed and appends it to `workspace.default.live_station_status`,
# with the same column names as silver. Station information is appended once per day to
# `workspace.default.live_station_info`. 

from datetime import datetime, timezone
import pandas as pd
import requests
from pyspark.sql import functions as F

ROOT = "https://gbfs.citibikenyc.com/gbfs/gbfs.json"
UA = {"User-Agent": "CUNY-SPS-DataScience-Archiver/1.0 (george.hagstrom@cuny.edu)"}
STATUS_TABLE = "workspace.default.live_station_status"
INFO_TABLE = "workspace.default.live_station_info"

spark.conf.set("spark.sql.session.timeZone", "UTC")


def get_json(url):
    response = requests.get(url, headers=UA, timeout=30)
    response.raise_for_status()
    return response.json()


feeds = {feed["name"]: feed["url"] for feed in get_json(ROOT)["data"]["en"]["feeds"]}
fetched_at = datetime.now(timezone.utc).replace(tzinfo=None)

# Station status: one row per station matching the silver format
status = pd.DataFrame(get_json(feeds["station_status"])["data"]["stations"])
status_frame = pd.DataFrame({
    "station_id": status["station_id"].astype(str),
    "fetched_at": fetched_at,
    "station_reported_at": pd.to_datetime(
        status["last_reported"].where(status["last_reported"] > 86400), unit="s"), # Same issue as we encountered in the main site. 86400 flags an inactive station
    "num_bikes_available": status["num_bikes_available"].astype("Int64"),
    "num_ebikes_available": status.get("num_ebikes_available", pd.Series(dtype="float")).astype("Int64"),
    "num_bikes_disabled": status["num_bikes_disabled"].astype("Int64"),
    "num_docks_available": status["num_docks_available"].astype("Int64"),
    "num_docks_disabled": status["num_docks_disabled"].astype("Int64"),
    "is_installed": status["is_installed"].astype(int).astype(bool),
    "is_renting": status["is_renting"].astype(int).astype(bool),
    "is_returning": status["is_returning"].astype(int).astype(bool),
})
spark.createDataFrame(status_frame).write.mode("append").saveAsTable(STATUS_TABLE)
print(f"{len(status_frame):,} status rows appended at {fetched_at:%Y-%m-%d %H:%M:%S} UTC")

# Station information: a snapshot once per day is sufficient
need_info = not spark.catalog.tableExists(INFO_TABLE) or (
    spark.table(INFO_TABLE).agg(F.max(F.to_date("fetched_at"))).first()[0] != fetched_at.date()
)
if need_info:
    info = pd.DataFrame(get_json(feeds["station_information"])["data"]["stations"])
    info_frame = pd.DataFrame({
        "station_id": info["station_id"].astype(str),
        "fetched_at": fetched_at,
        "name": info["name"].astype(str),
        "capacity": info["capacity"].astype("Int64"),
        "lat": info["lat"].astype(float),
        "lon": info["lon"].astype(float),
    })
    spark.createDataFrame(info_frame).write.mode("append").saveAsTable(INFO_TABLE)
    print(f"{len(info_frame):,} station information rows appended")